# 02 — Koalas / PySpark Benchmark

This notebook replicates the **Koalas (PySpark)** side of the experiment.

> **Note on Koalas:** Since PySpark 3.2 (released Nov 2021), the Koalas API is
> bundled inside PySpark as `pyspark.pandas` (also called *pandas API on Spark*).
> The original `databricks.koalas` package is now deprecated.  We use
> `pyspark.pandas` throughout, which is API-compatible.

We run the same **15 operations** and **3 scenarios** as the Dask notebook:

| Scenario | Code |
|---|---|
| **Standard** | `operations(df)` |
| **Filtered** | `operations(df[(df.tip_amt >= 1) & (df.tip_amt < 5)])` |
| **Filtered + Cached** | `df.spark.cache()` → `operations(df_cached)` |

## 1  Setup

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd

from config import parquet_path, RESULTS_DIR, TIP_FILTER_MIN, TIP_FILTER_MAX
from src.benchmark_utils import BenchmarkTimer, build_results_table, print_table
from src.operations_koalas import run_operations, read_parquet_timing, cache_dataframe

os.makedirs(RESULTS_DIR, exist_ok=True)

### 1a  Start PySpark session

**Local mode**: `master='local[*]'` uses all available CPU cores.  
**Dataproc**: PySpark session is already available; skip this cell and use `SparkSession.getActiveSession()`.

In [ ]:
from pyspark.sql import SparkSession

# Detect if we're on Dataproc (SparkSession already exists)
existing = SparkSession.getActiveSession()

if existing is None:
    import multiprocessing
    cores = multiprocessing.cpu_count()
    spark = (
        SparkSession.builder
        .master(f'local[{cores}]')
        .appName('cdle-koalas-benchmark')
        .config('spark.driver.memory', '8g')
        .config('spark.executor.memory', '8g')
        .config('spark.sql.shuffle.partitions', str(cores * 4))
        # Required for pyspark.pandas
        .config('spark.sql.execution.arrow.pyspark.enabled', 'true')
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel('WARN')
    print(f'New local[{cores}] SparkSession')
else:
    spark = existing
    print('Using existing SparkSession (Dataproc mode)')

print('Spark version:', spark.version)

In [ ]:
# Import pyspark.pandas (replaces koalas)
import pyspark.pandas as ps

# Tell pyspark.pandas to use the existing session
ps.set_option('compute.default_index_type', 'sequence')

print('pyspark.pandas version:', ps.__version__)

## 2  Load data

In [ ]:
DATA_PATH = parquet_path('yellow_taxi.parquet')
print(f'Reading: {DATA_PATH}')

# Time the read_parquet operation
t_read = read_parquet_timing(DATA_PATH, spark, label='koalas_standard')

# Load for the rest of the benchmark
df = ps.read_parquet(DATA_PATH)
print(f'Columns: {list(df.columns)}')
print(f'Rows   : {len(df):,}')
df.head(3)

## 3  Scenario 1 — Standard operations

In [ ]:
print('=== SCENARIO 1: STANDARD OPERATIONS ===')
t_std = run_operations(df, label='koalas_standard')
t_std._results['read_parquet'] = t_read._results.get('read_parquet', float('nan'))
print_table(t_std.to_series().to_frame(), title='Koalas — Standard operations')

## 4  Scenario 2 — Operations with filtering

```python
# Filtering is lazily composed into the Spark plan
df_filtered = df[(df.tip_amt >= 1) & (df.tip_amt < 5)]
operations(df_filtered)
```

Spark's **Catalyst optimizer** pushes the filter down to the Parquet scan level,
so only the relevant row groups are read from disk.

In [ ]:
df_filtered = df[(df['tip_amt'] >= TIP_FILTER_MIN) & (df['tip_amt'] < TIP_FILTER_MAX)]

print('=== SCENARIO 2: OPERATIONS WITH FILTERING ===')
t_flt = run_operations(df_filtered, label='koalas_filtered')
print_table(t_flt.to_series().to_frame(), title='Koalas — Filtered operations')

## 5  Scenario 3 — Operations with filtering and caching

```python
# Koalas / pyspark.pandas caching
df_cached = df[(df.tip_amt >= 1) & (df.tip_amt < 5)]
df_cached.spark.cache()                       # mark for caching
df_cached.spark.apply(lambda sdf: sdf.count()) # materialise into executor memory
operations(df_cached)
```

In [ ]:
print('Caching filtered DataFrame into Spark executor memory …')
df_cached = cache_dataframe(df_filtered)
print('Cache ready.')

print('=== SCENARIO 3: OPERATIONS WITH FILTERING + CACHING ===')
t_cache = run_operations(df_cached, label='koalas_cached')
print_table(t_cache.to_series().to_frame(), title='Koalas — Filtered+Cached operations')

## 6  Results summary

In [ ]:
import matplotlib.pyplot as plt

results_df = build_results_table(t_std, t_flt, t_cache)
print_table(results_df, title='KOALAS BENCHMARK — All scenarios (seconds)')

csv_path = os.path.join(RESULTS_DIR, 'koalas_benchmark.csv')
results_df.to_csv(csv_path)
print(f'Saved: {csv_path}')

In [ ]:
ax = results_df.plot(
    kind='bar', figsize=(14, 6),
    title='Koalas — Elapsed time per operation (seconds)',
    ylabel='Time (s)', rot=45
)
ax.legend(['Standard', 'Filtered', 'Filtered+Cached'])
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'koalas_benchmark.png'), dpi=120)
plt.show()

## 7  Notes on Koalas / pyspark.pandas execution

### Lazy evaluation & Catalyst optimizer
Every `pyspark.pandas` expression is translated into a Spark SQL logical plan.
Before execution, Catalyst applies:
- **Filter pushdown**: moves `WHERE` conditions to the file scan, reading only needed row-groups.
- **Column pruning**: reads only the columns required by the query.
- **BroadcastHashJoin**: when one side of a join is small, Spark broadcasts it to all executors instead of shuffling.
- **Code generation**: Spark SQL compiles each query to optimised JVM bytecode at runtime.

### Caching vs Dask
| | Dask | Koalas/Spark |
|---|---|---|
| Cache command | `client.persist(df)` | `df.spark.cache()` |
| Wait for materialisation | `wait(df)` | `df.spark.apply(lambda sdf: sdf.count())` |
| Storage location | Worker process RAM | Spark executor memory (off-heap option available) |

In [ ]:
spark.stop()